# PARTE 2. COMPRENSIÓN Y PREPARACIÓN DE LOS DATOS

In [23]:
import pandas as pd
import numpy as np
import os
import warnings
import argparse
from pathlib import Path
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [14]:
import pandas as pd
import numpy as np

def analizar_comprension_datos(file_path):
    print(f"{'='*60}")
    print(f"ANÁLISIS EXPLORATORIO DE CALIDAD DE DATOS: {file_path}")
    print(f"{'='*60}\n")
    
    # Cargar el archivo Excel
    try:
        xls = pd.ExcelFile(file_path)
        hojas = xls.sheet_names
        print(f"Estaciones (hojas) detectadas: {hojas}\n")
    except Exception as e:
        print(f"Error al leer el archivo: {e}")
        return None
        
    # Consolidar los datos de toda las estaciones en un solo df
    df_list = []
    for hoja in hojas:
        df_temp = pd.read_excel(xls, sheet_name=hoja)
        df_temp['Estacion'] = hoja # Estación como variable categórica
        df_list.append(df_temp)
        
    df = pd.concat(df_list, ignore_index=True)
    
    # 1A. Dimensión del dataset
    print("--- 1A. DIMENSIÓN DEL DATASET ---")
    registros, columnas = df.shape
    print(f"Cantidad total de registros: {registros}")
    print(f"Cantidad total de columnas: {columnas}\n")
    
    # 1B. Descripción de variables
    print("--- 1B. DESCRIPCIÓN DE VARIABLES ---")
    resumen_vars = []
    
    for col in df.columns:
        # Determinar el tipo de dato
        if pd.api.types.is_numeric_dtype(df[col]):
            tipo = "Numérico"
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            tipo = "Fecha/Hora"
        else:
            tipo = "Categórico / Texto"
            
        nulos = df[col].isnull().sum()
        pct_nulos = (nulos / registros) * 100
        
        # Determinar valores posibles o rango
        if tipo == "Numérico":
            posibles_valores = f"Rango: [{df[col].min()} a {df[col].max()}]"
        elif tipo == "Fecha/Hora":
            posibles_valores = f"Desde {df[col].min()} hasta {df[col].max()}"
        else:
            # Para variables categóricas, mostrar los primeros valores únicos
            unicos = df[col].dropna().unique()
            posibles_valores = f"Valores: {list(unicos[:5])}" + ("..." if len(unicos) > 5 else "")
            
        resumen_vars.append({
            "Variable": col,
            "Tipo": tipo,
            "Nulos Totales": f"{nulos} ({pct_nulos:.2f}%)",
            "Valores Posibles": posibles_valores
        })
        
    df_resumen = pd.DataFrame(resumen_vars)
    # Ajustamos opciones de pandas para imprimir la tabla completa
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_rows', None)
    print(df_resumen.to_string(index=False))
    print("\n")
    
    # 1C. Calidad de los datos
    print("--- 1C. CALIDAD DE LOS DATOS ---")
    
    # Resumen de faltantes
    total_nulos = df.isnull().sum().sum()
    total_celdas = registros * columnas
    print(f"Valores faltantes en todo el dataset: {total_nulos} de {total_celdas} celdas ({(total_nulos/total_celdas)*100:.2f}%)\n")
    
    # Identificación de valores erróneos
    print("Verificación de valores espurios o erróneos (físicamente imposibles):")
    errores_encontrados = False
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    
    for col in cols_numericas:
        min_val = df[col].min()
        max_val = df[col].max()
        
        # Regla 1: Las concentraciones de contaminantes y velocidades no pueden ser negativas
        if min_val < 0 and col not in ['TOUT', 'PRS']: # La temperatura sí puede bajar de 0, la presión depende de cómo se mida
            print(f"Error en {col}: Se encontraron valores negativos (mínimo = {min_val}).")
            errores_encontrados = True
            
        # Regla 2: Humedad relativa (RH) no puede ser mayor a 100 ni menor a 0
        if col == 'RH':
            if max_val > 100 or min_val < 0:
                print(f"Error en {col} (Humedad): Valores fuera del rango 0-100% (min={min_val}, max={max_val}).")
                errores_encontrados = True
                
        # Regla 3: Dirección del viento (WDR) debe estar entre 0 y 360 grados
        if col == 'WDR':
            if max_val > 360 or min_val < 0:
                print(f"Error en {col} (Dir. Viento): Valores fuera del rango 0-360° (min={min_val}, max={max_val}).")
                errores_encontrados = True

    if not errores_encontrados:
        print(f"No se detectaron valores negativos ni rangos climatológicos erróneos básicos.\n")
    return df

In [15]:
years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

for year in years:
    analizar_comprension_datos(f'BD {year}.xlsx')

ANÁLISIS EXPLORATORIO DE CALIDAD DE DATOS: BD 2020.xlsx

Estaciones (hojas) detectadas: ['SE', 'CE', 'SO', 'NE2', 'SE2', 'SE3', 'NE', 'NO', 'NO2', 'NTE', 'NTE2', 'SO2', 'SUR']

--- 1A. DIMENSIÓN DEL DATASET ---
Cantidad total de registros: 114102
Cantidad total de columnas: 17

--- 1B. DESCRIPCIÓN DE VARIABLES ---
    Variable               Tipo  Nulos Totales                                    Valores Posibles
Fecha y hora         Fecha/Hora      0 (0.00%) Desde 2020-01-01 00:00:00 hasta 2020-12-31 23:00:00
          CO           Numérico 49801 (43.65%)                               Rango: [0.05 a 17.71]
          NO           Numérico 69722 (61.10%)                                Rango: [0.5 a 500.0]
         NO2           Numérico 79349 (69.54%)                                Rango: [0.0 a 188.6]
         NOX           Numérico 74897 (65.64%)                                Rango: [0.5 a 500.0]
          O3           Numérico 69107 (60.57%)                                Rango: [1.0 

In [ ]:
def consolidar_archivos_bd():
    # Años disponibles a procesar
    años = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
    dataframes_completos = []

    for año in años:
        nombre_archivo = f'BD {año}.xlsx'
        
        # Verificamos si el archivo existe
        if os.path.exists(nombre_archivo):
            print(f"Procesando {nombre_archivo}...")
            
            # Leemos todas las hojas del archivo
            diccionario_hojas = pd.read_excel(nombre_archivo, sheet_name=None)
            
            for nombre_estacion, df_hoja in diccionario_hojas.items():
                # Homologación de columna de fechas
                if 'date' in df_hoja.columns:
                    df_hoja = df_hoja.rename(columns={'date': 'Fecha y hora'})
                
                # Agregamos metadatos de trazabilidad
                df_hoja['Estacion'] = nombre_estacion
                df_hoja['Archivo_Origen'] = nombre_archivo
                
                dataframes_completos.append(df_hoja)
        else:
            print(f" [!] Advertencia: No se encontró el archivo '{nombre_archivo}'")

    # Consolidación final
    if dataframes_completos:
        df_final = pd.concat(dataframes_completos, ignore_index=True)
        print("\n¡Consolidación terminada con éxito!")
        print(f"Dimensiones del dataset final: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
        return df_final
    else:
        print("\nNo se pudo consolidar ningún archivo.")
        return pd.DataFrame()

# Ejecución:
df_historico = consolidar_archivos_bd()

Procesando BD 2020.xlsx...
Procesando BD 2021.xlsx...
Procesando BD 2022.xlsx...
Procesando BD 2023.xlsx...
Procesando BD 2024.xlsx...
Procesando BD 2025.xlsx...
Procesando BD 2026.xlsx...

¡Consolidación terminada con éxito!
Dimensiones del dataset final: 830908 filas x 18 columnas


In [ ]:
def limpiar_y_resumir_datos_v2(df, col_fecha='Fecha y hora'):
    df_clean = df.copy()
    
    # Eliminación de fechas estrictamente NaN
    total_inicial = len(df_clean)
    filas_fecha_nula = df_clean[col_fecha].isnull().sum()
    df_clean = df_clean.dropna(subset=[col_fecha])
    
    # Extraemos el año solo para el resumen
    años_extraidos = pd.to_datetime(df_clean[col_fecha], errors='coerce').dt.year
    df_clean['Año_str'] = años_extraidos.fillna(-1).astype(int).astype(str).replace('-1', 'Desconocido')
    años_unicos = df_clean['Año_str'].unique()
    
    # Imputación de valores
    cols_no_negativas = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 
                         'PRS', 'RAINF', 'RH', 'SO2', 'SR', 'WSR', 'WDR']
    
    cambios_global = {}
    cambios_año = {año: {} for año in años_unicos}
    
    for col in cols_no_negativas:
        if col in df_clean.columns:
            mask_extremos = df_clean[col] < -999
            mask_regulares = (df_clean[col] < 0) & (df_clean[col] >= -999)
            
            n_extremos = mask_extremos.sum()
            n_regulares = mask_regulares.sum()
            
            cambios_global[col] = {'extremos': n_extremos, 'regulares': n_regulares}
            
            if n_extremos > 0 or n_regulares > 0:
                # Conteo por año
                for año in años_unicos:
                    mask_año = (df_clean['Año_str'] == año)
                    ext_año = (mask_extremos & mask_año).sum()
                    reg_año = (mask_regulares & mask_año).sum()
                    
                    if ext_año > 0 or reg_año > 0:
                        cambios_año[año][col] = {'extremos': ext_año, 'regulares': reg_año}
                
                df_clean.loc[mask_extremos | mask_regulares, col] = np.nan
                
    # Quitar duplicados y verificar columnas
    columnas_antes = set(df_clean.columns)
    filas_antes_dup = len(df_clean)
    
    df_clean = df_clean.drop_duplicates()
    
    filas_duplicadas_removidas = filas_antes_dup - len(df_clean)
    
    # Verificar si alguna columna desapareció en el proceso general
    df_clean = df_clean.drop(columns=['Año_str'])
    columnas_despues = set(df_clean.columns)
    columnas_removidas = list(columnas_antes - columnas_despues - {'Año_str'})
    
    # Generación del Reporte
    print(f"{'='*65}")
    print("REPORTE DE LIMPIEZA DE DATOS (CRISP-DM V2)")
    print(f"{'='*65}")
    print(f"-> Filas iniciales: {total_inicial}")
    print(f"-> Filas eliminadas por NaN explícito en '{col_fecha}': {filas_fecha_nula}")
    print(f"-> Filas eliminadas por duplicidad (drop_duplicates): {filas_duplicadas_removidas}")
    
    if columnas_removidas:
        print(f"-> [!] Columnas removidas en el proceso: {columnas_removidas}")
    else:
        print("-> Columnas removidas: Ninguna (drop_duplicates opera sobre filas, no afectó columnas).")
        
    print(f"-> Filas finales conservadas: {len(df_clean)}\n")
    
    print("--- CAMBIOS GLOBALES (Valores Negativos -> NaN) ---")
    hubo_cambios = False
    for col, conteos in cambios_global.items():
        ext = conteos['extremos']
        reg = conteos['regulares']
        if ext > 0 or reg > 0:
            print(f" - {col:<5}: {ext:<5} valores < -999  |  {reg:<5} valores entre -999 y < 0")
            hubo_cambios = True
    if not hubo_cambios:
        print(" No se detectaron valores negativos espurios globalmente.")
        
    print("\n--- CAMBIOS DETALLADOS POR AÑO ---")
    años_ordenados = sorted([a for a in años_unicos if a != 'Desconocido'])
    if 'Desconocido' in años_unicos:
        años_ordenados.append('Desconocido')
        
    for año in años_ordenados:
        cambios = cambios_año[año]
        if cambios:
            print(f"\n[ Año {año} ]")
            for col, conteos in cambios.items():
                print(f"   - {col:<5}: {conteos['extremos']:<4} (< -999)  |  {conteos['regulares']:<4} (>= -999 y < 0)")

    return df_clean

# Ejecución de ejemplo:
df_final = limpiar_y_resumir_datos_v2(df_historico)

REPORTE DE LIMPIEZA DE DATOS (CRISP-DM V2)
-> Filas iniciales: 830908
-> Filas eliminadas por NaN explícito en 'Fecha y hora': 4
-> Filas eliminadas por duplicidad (drop_duplicates): 0
-> Columnas removidas: Ninguna (drop_duplicates opera sobre filas, no afectó columnas).
-> Filas finales conservadas: 830904

--- CAMBIOS GLOBALES (Valores Negativos -> NaN) ---
 - NOX  : 2     valores < -999  |  0     valores entre -999 y < 0
 - RAINF: 1     valores < -999  |  0     valores entre -999 y < 0
 - RH   : 9     valores < -999  |  0     valores entre -999 y < 0
 - SR   : 11    valores < -999  |  270   valores entre -999 y < 0
 - WSR  : 1     valores < -999  |  0     valores entre -999 y < 0
 - WDR  : 1     valores < -999  |  0     valores entre -999 y < 0

--- CAMBIOS DETALLADOS POR AÑO ---

[ Año 2021 ]
   - SR   : 10   (< -999)  |  16   (>= -999 y < 0)

[ Año 2022 ]
   - RH   : 2    (< -999)  |  0    (>= -999 y < 0)
   - SR   : 1    (< -999)  |  86   (>= -999 y < 0)

[ Año 2023 ]
   - SR   

In [ ]:
# Importamos el archivo sima.py que debe estar en tu misma carpeta
import sima 

def ejecutar_imputacion_sima(df_limpio):
    print("Preparando datos para la arquitectura SIMA...")
    
    # Renombrar columnas para cumplir con las exigencias de sima.py
    df_sima = df_limpio.rename(columns={
        'Estacion': 'ID',
        'Fecha y hora': 'time'
    })
    
    # Guardar temporalmente como CS
    ruta_input = Path("data_temporal_sima.csv")
    df_sima.to_csv(ruta_input, index=False)
    print(f"Archivo temporal guardado en: {ruta_input}")
    
    # Configurar los hiperparámetros simulando los argumentos de consola
    args = argparse.Namespace(
        command='run',
        input=ruta_input,
        device='auto',             # Usa GPU (cuda) si está disponible
        batch_size=32,
        workers=0,                 
        seed=42,
        train_fraction=0.8,
        min_observed=0.2,
        eval_stride=168,
        seq_len=168,               # Ventana temporal de 168 horas
        hidden_size=64,
        n_heads=4,
        n_layers=2,
        diffusion_steps=50,
        dropout=0.1,
        residual_scale=0.1,
        epochs=10,                 
        learning_rate=1e-3,
        mask_ratio=0.2,
        train_stride=24,
        checkpoint=Path("models/sima.pt"),
        output_dir=Path("outputs_sima"),
        samples=20,
        max_batches=20,
        impute_stride=84
    )
    
    # Ejecutar el pipeline completo de sima.py (Train -> Evaluate -> Impute)
    print("Iniciando el entrenamiento e imputación. Esto puede tomar tiempo...")
    sima.command_run(args)
    
    # Cargar el resultado imputado de vuelta al notebook
    ruta_imputado = Path("outputs_sima/sima_dataset_imputado_completo.csv")
    
    if ruta_imputado.exists():
        df_imputado = pd.read_csv(ruta_imputado)
        
        df_imputado = df_imputado.rename(columns={'ID': 'Estacion', 'time': 'Fecha y hora'})
        print(f"\n¡Imputación completada! Dataset final: {df_imputado.shape}")
        
        # Limpieza del archivo temporal
        ruta_input.unlink(missing_ok=True)
        
        return df_imputado
    else:
        print("\n[!] Error: No se encontró el archivo resultante imputado.")
        return None

df_completado = ejecutar_imputacion_sima(df_final)

Preparando datos para la arquitectura SIMA...
Archivo temporal guardado en: data_temporal_sima.csv
Iniciando el entrenamiento e imputación. Esto puede tomar tiempo...

¡Imputación completada! Dataset final: (831095, 17)
